# Word Sense Disambiguasi (WSD) Esai Biologi dengan Simplified Lesk

Esai siswa memuat banyak kata ambigu: satu kata, beberapa arti di KBBI.
Notebook ini menentukan arti (word sense) kata ambigu berdasarkan **konteks
kalimat** dengan algoritma *simplified Lesk*, lalu membandingkan pilihan sense
pada **kunci jawaban guru** dan **jawaban siswa** sebagai fitur pendukung
*automatic essay scoring* (AES).

Alur kerja: baca data → kamus KBBI → Lesk → perbandingan guru vs siswa →
studi kasus → penandaan warna pada jawaban lengkap.

In [ ]:
# sumber data: ground truth WSD & dataset esai
# di Colab jalankan sekali: !pip install PySastrawi -q
import pandas as pd
import re
import requests
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

df = None
for enc in ("utf-8-sig", "cp1252", "latin-1"):
    try:
        df = pd.read_csv("groundtruth_selesai.csv", sep=None, engine="python", encoding=enc)
        break
    except UnicodeDecodeError:
        pass
df["Sense_Benar"] = pd.to_numeric(df["Sense_Benar"], errors="coerce")
df = df.dropna(subset=["Sense_Benar"]).reset_index(drop=True)
data = pd.read_excel("Dataset Biologi.xlsx")

print("ground truth :", len(df), "token |", df["kata_token"].nunique(), "kata unik")
print("esai siswa   :", len(data), "baris |", data["Nama"].nunique(), "siswa |",
      data["Kode_Soal"].nunique(), "soal")

In [ ]:
# kamus KBBI: kata -> daftar gloss (arti) dari ground truth
def baca_gloss(teks):
    """pecah '1) ... | 2) ...' menjadi list gloss."""
    bagian = re.split(r"\s*\|\s*(?=\d+\)\s)", str(teks))
    return [re.sub(r"^\d+\)\s*", "", b).strip() for b in bagian if b.strip()]

KAMUS_KBBI = {}
for _, baris in df.iterrows():
    KAMUS_KBBI.setdefault(baris["kata_token"], baca_gloss(baris["Daftar_Gloss"]))

kata_ambigu = {k for k, v in KAMUS_KBBI.items() if len(v) > 1}
print("kata ambigu (sense > 1) :", len(kata_ambigu), "kata")
print()
print("contoh 'tegang':")
for i, g in enumerate(KAMUS_KBBI["tegang"], 1):
    print(f"  {i}) {g[:70]}")

## Simplified Lesk

Sense dipilih dari gloss yang paling banyak beriraman kata dengan kalimat
(stopword dibuang, konteks diambil unik):

- `skor(sense) = jumlah kata konteks yang juga ada di gloss sense tersebut`
- sense terpilih = skor terbesar; **kalau skor nol semua → sense 1** (arti pertama KBBI)

In [ ]:
STOP = set(StopWordRemoverFactory().get_stop_words())

def bersihkan(teks):
    """ambil kata-katanya saja (tanpa tanda baca & angka, huruf kecil)."""
    return re.findall(r"[a-z]+", str(teks).lower())

def lesk(kalimat, kata_target):
    """kembalikan nomor sense terpilih."""
    if kata_target not in KAMUS_KBBI:
        return None
    konteks = [k for k in dict.fromkeys(bersihkan(kalimat)) if k not in STOP]
    skor = [sum(1 for k in konteks if k != kata_target and k in g.split())
            for g in KAMUS_KBBI[kata_target]]
    return skor.index(max(skor)) + 1

benar = sum(lesk(b["Kunci_Jawaban"], b["kata_token"]) == b["Sense_Benar"]
            for _, b in df.iterrows())
print(f"akurasi simplified lesk di ground truth : {benar}/{len(df)} = {benar/len(df)*100:.1f}%")

In [ ]:
# analisis: akurasi saat gloss sense yang benar termasuk istilah biologi vs umum
ISTILAH_BIO = {"sel", "organisme", "protein", "molekul", "membran", "energi", "ion",
               "zat", "cairan", "organ", "jaringan", "enzim", "hormon", "darah",
               "tumbuhan", "hewan", "bakteri", "virus", "metabolisme", "nutrisi", "gas"}

def kategori_gloss(gloss):
    return "biologi" if any(t in gloss.split() for t in ISTILAH_BIO) else "umum"

b_bio = t_bio = b_umum = t_umum = 0
for _, baris in df.iterrows():
    ok = lesk(baris["Kunci_Jawaban"], baris["kata_token"]) == baris["Sense_Benar"]
    if kategori_gloss(KAMUS_KBBI[baris["kata_token"]][int(baris["Sense_Benar"]) - 1]) == "biologi":
        b_bio += ok; t_bio += 1
    else:
        b_umum += ok; t_umum += 1

print(f"akurasi saat sense benar bergloss BIOLOGI : {b_bio}/{t_bio} = {b_bio/t_bio*100:.1f}%")
print(f"akurasi saat sense benar bergloss UMUM    : {b_umum}/{t_umum} = {b_umum/t_umum*100:.1f}%")
print(">> lesk lebih akurat pada gloss beristilah biologi; gloss umum jadi sumber kesalahan utama")

In [ ]:
# perbandingan sense guru vs siswa di seluruh data
def bandingkan_esai(baris):
    """bandingkan sense guru vs siswa untuk kata ambigu yang sama."""
    g = bersihkan(baris["Kunci Guru"])
    s = bersihkan(baris["Jawaban Siswa"])
    hasil = []
    for kata in sorted(set(g) & set(s) & kata_ambigu):
        if kata in STOP:
            continue
        hasil.append((kata, lesk(" ".join(g), kata), lesk(" ".join(s), kata)))
    return hasil

hasil = []
for _, baris in data.iterrows():
    for kata, sg, ss in bandingkan_esai(baris):
        hasil.append({"Kode_Soal": baris["Kode_Soal"], "kata": kata,
                      "sense_guru": sg, "sense_siswa": ss,
                      "status": "SESUAI" if sg == ss else "TIDAK SESUAI"})
perbandingan = pd.DataFrame(hasil)
n, n_ok = len(perbandingan), (perbandingan["status"] == "SESUAI").sum()
print("total kata ambigu yang dibandingkan :", n)
print(f"SESUAI       : {n_ok} ({n_ok/n*100:.1f}%)")
print(f"TIDAK SESUAI : {n - n_ok} ({(n - n_ok)/n*100:.1f}%)")
print()
print("10 kata yang paling sering beda dengan guru:")
print(perbandingan[perbandingan["status"] == "TIDAK SESUAI"]["kata"]
      .value_counts().head(10).to_string())

In [ ]:
# simpan rekap + sheet Catatan (definisi penilaian SESUAI)
perbandingan["keterangan"] = [
    f"kunci = word sense guru (sense {r['sense_guru']}); "
    f"siswa sense {r['sense_siswa']} -> {r['status'].lower()}"
    for _, r in perbandingan.iterrows()]

with pd.ExcelWriter("hasil_wsd_guru_vs_siswa.xlsx") as w:
    perbandingan.to_excel(w, index=False, sheet_name="guru_vs_siswa")
    pd.DataFrame({"catatan": [
        "KATA DISEBUT WORD SENSE-NYA BENAR/SESUAI KARENA YANG DIJADIKAN KUNCI",
        "ADALAH WORD SENSE DARI JAWABAN GURU (kolom sense_guru).",
        "sense_siswa dibandingkan ke sense_guru untuk kata yang sama:",
        "nomor sense sama -> SESUAI, beda -> TIDAK SESUAI.",
    ]}).to_excel(w, index=False, sheet_name="Catatan")
perbandingan.to_csv("hasil_wsd_guru_vs_siswa.csv", index=False, sep=";")
print("disimpan : hasil_wsd_guru_vs_siswa.xlsx (+ sheet Catatan) & versi csv ';'")

## Studi kasus: `tegang` (soal 17) dan `membran` (soal 20)

Dua kata ini polanya berkebalikan, jadi bagus untuk melihat cara kerja lesk:

- `tegang` → konteks **guru** yang kena gloss (kata *kaku*) → guru sense 2, siswa default
- `membran` → konteks **siswa** yang kena gloss (kata *molekul*) → siswa sense 3, guru default

Sense diambil langsung dari **KBBI online**; kata dinilai SESUAI bila
sense siswa = sense guru.

In [ ]:
# sambungan KBBI online: arti/sense diambil langsung dari API
CACHE_KBBI = {}

def ambil_kbbi_online(kata):
    """return (daftar arti, sumber); kalau jaringan gagal -> kamus mini."""
    if kata in CACHE_KBBI:
        return CACHE_KBBI[kata], "online (cache)"
    try:
        res = requests.get(f"https://kbbi.raf555.dev/api/v1/entry/{kata}", timeout=30).json()
        arti = [d["definition"] for e in res["entries"] for d in e["definitions"]]
        if not arti:
            raise ValueError("tidak ada arti")
        CACHE_KBBI[kata] = arti
        return arti, "online"
    except Exception:
        return KAMUS_KBBI.get(kata, []), "kamus mini (offline)"

kata_uji = "tegang"
print("URL :", f"https://kbbi.raf555.dev/api/v1/entry/{kata_uji}")
arti, sumber = ambil_kbbi_online(kata_uji)
print(f"KBBI online ({sumber}) : {len(arti)} arti untuk '{kata_uji}'")
for i, a in enumerate(arti, 1):
    print(f"  {i}) {a[:74]}")

In [ ]:
# penelusuran 1 kata: kalimat utuh guru & siswa -> skor -> nomor sense
def telusuri(kata, kode_soal, nama_siswa):
    """telusuri 1 kata: kalimat guru & siswa -> kata pencocok -> nomor sense."""
    daftar_sense, sumber = ambil_kbbi_online(kata)
    baris_soal = data[data["Kode_Soal"] == kode_soal]
    kunci = baris_soal.iloc[0]["Kunci Guru"]
    jawab = baris_soal[baris_soal["Nama"] == nama_siswa].iloc[0]["Jawaban Siswa"]
    print("=" * 68)
    print(f"KATA UJI '{kata}' (soal {kode_soal} | siswa: {nama_siswa}) | sense: {sumber}")
    print("=" * 68)
    pilihan = {}
    for label, teks in [("GURU", str(kunci)), ("SISWA", str(jawab))]:
        print(f"KALIMAT {label} (utuh):")
        print(f'"{teks}"')
        konteks = list(dict.fromkeys([k for k in bersihkan(teks) if k not in STOP]))
        skor = [sum(1 for k in konteks if k != kata and k in g.split())
                for g in daftar_sense]
        pilih = skor.index(max(skor)) + 1
        pilihan[label] = pilih
        kena = [k for k in konteks if k in daftar_sense[pilih - 1].split()]
        print(f"skor per sense {skor} -> sense {pilih}")
        print("kata pencocok :", kena if kena
              else "tidak ada (skor nol semua -> default sense 1)")
        print("-" * 68)
    status = "SESUAI" if pilihan["GURU"] == pilihan["SISWA"] else "TIDAK SESUAI"
    print(f"KESIMPULAN : sense_guru = {pilihan['GURU']}, "
          f"sense_siswa = {pilihan['SISWA']} -> {status}")
    return pilihan

telusuri("tegang", 17, "Afif Ardliansyah Nur Holis")

In [ ]:
# kasus kedua (polanya kebalikan dari tegang)
telusuri("membran", 20, "Nabilah Aushaf Ramadhani")

In [ ]:
# studi kasus untuk SEMUA siswa pada soal yang sama
def tabel_studi_kasus(kata, kode_soal):
    """telusuri 1 kata ambigu untuk SEMUA siswa di 1 soal."""
    daftar_sense, sumber = ambil_kbbi_online(kata)
    baris_soal = data[data["Kode_Soal"] == kode_soal]
    kunci = baris_soal.iloc[0]["Kunci Guru"]
    konteks_g = list(dict.fromkeys([k for k in bersihkan(kunci) if k not in STOP]))
    skor_g = [sum(1 for k in konteks_g if k != kata and k in g.split())
              for g in daftar_sense]
    sense_g = skor_g.index(max(skor_g)) + 1
    kena_g = [k for k in konteks_g if k in daftar_sense[sense_g - 1].split()]
    rows = []
    for _, b in baris_soal.iterrows():
        if kata not in bersihkan(b["Jawaban Siswa"]):
            continue
        konteks_s = list(dict.fromkeys(
            [k for k in bersihkan(b["Jawaban Siswa"]) if k not in STOP]))
        skor_s = [sum(1 for k in konteks_s if k != kata and k in g.split())
                  for g in daftar_sense]
        sense_s = skor_s.index(max(skor_s)) + 1
        kena_s = [k for k in konteks_s if k in daftar_sense[sense_s - 1].split()]
        rows.append({
            "Nama": b["Nama"], "Kode_Soal": kode_soal, "kata": kata,
            "kunci_jawaban_guru": str(kunci)[:120],
            "kata_pencocok_dari_guru": ", ".join(kena_g) if kena_g else "-",
            "jawaban_siswa": str(b["Jawaban Siswa"])[:120],
            "kata_pencocok_dari_siswa": ", ".join(kena_s) if kena_s else "-",
            "sumber_KBBI": sumber,
            "skor_guru": str(skor_g), "sense_guru": sense_g,
            "skor_siswa": str(skor_s), "sense_siswa": sense_s,
            "status": "SESUAI" if sense_s == sense_g else "TIDAK SESUAI",
            "keterangan": (f"kunci penilaian = word sense guru (sense {sense_g}, "
                           f"gloss KBBI online). pencocokan: dari kunci guru {kena_g}; "
                           f"dari jawaban siswa {kena_s}")})
    return pd.DataFrame(rows)

tabel_tegang = tabel_studi_kasus("tegang", 17)
tabel_membran = tabel_studi_kasus("membran", 20)
for nama, tabel in [("tegang", tabel_tegang), ("membran", tabel_membran)]:
    n_ok_kasus = (tabel["status"] == "SESUAI").sum()
    print(f"studi kasus {nama:8s}: {len(tabel)} siswa | SESUAI {n_ok_kasus} | "
          f"TIDAK SESUAI {len(tabel) - n_ok_kasus}")

In [ ]:
# simpan studi kasus: xlsx (telusuri + arti KBBI online) & csv pemisah ';'
def sheet_kbbi_online(kata):
    arti, _ = ambil_kbbi_online(kata)
    return pd.DataFrame({"sense": range(1, len(arti) + 1),
                         "arti_dari_KBBI_online": arti})

with pd.ExcelWriter("studi_kasus_tegang.xlsx") as w:
    tabel_tegang.to_excel(w, index=False, sheet_name="telusuri_tegang")
    sheet_kbbi_online("tegang").to_excel(w, index=False, sheet_name="KBBI_online")
with pd.ExcelWriter("studi_kasus_membran.xlsx") as w:
    tabel_membran.to_excel(w, index=False, sheet_name="telusuri_membran")
    sheet_kbbi_online("membran").to_excel(w, index=False, sheet_name="KBBI_online")
tabel_tegang.to_csv("studi_kasus_tegang.csv", index=False, sep=";")
tabel_membran.to_csv("studi_kasus_membran.csv", index=False, sep=";")
print(tabel_tegang.head(3).to_string(index=False, max_colwidth=24))
print()
print("disimpan : studi_kasus_tegang.xlsx/.csv & studi_kasus_membran.xlsx/.csv")

## Penandaan warna pada jawaban lengkap

Setiap kata pada **jawaban lengkap** (dataset awal) diberi warna sesuai sense
yang ia masuki: kata ada di gloss sense ke-*n* → warna ke-*n*. Nomor sense =
**arti ke-n di KBBI online**, jadi legenda warna langsung menunjukkan artinya.
**Kunci jawaban guru diberi warna dengan cara yang sama**, sehingga terlihat
bagian kata yang menentukan sense guru/siswa.

Keterangan warna: kata target = pink; warna lain = sense yang dimasuki
(teks putih di atas warna gelap agar tidak bertabrakan dengan teks);
kata tanpa warna (hitam) = tidak masuk gloss mana pun → tidak memberi skor.

In [ ]:
# warna per sense: gelap + teks putih (aman di tema terang maupun gelap)
WARNA_SENSE = {1: "#B26A00", 2: "#E65100", 3: "#01579B",
               4: "#1B5E20", 5: "#4A148C", 6: "#880E4F", 7: "#BF360C"}
WARNA_TARGET = "#C2185B"

def pecah_warna(teks, kata_target):
    """pecah kalimat jadi list (potongan, nomor sense yang kena)."""
    gloss, _ = ambil_kbbi_online(kata_target)
    potongan = []
    for token in re.findall(r"[a-z]+|[^a-z]+", str(teks).lower()):
        if not token.isalpha():
            potongan.append((token, None))
        elif token == kata_target:
            potongan.append((token, "TARGET"))
        elif token in STOP:
            potongan.append((token, None))
        else:
            nomor = None
            for n, g in enumerate(gloss, 1):
                if token in g.split():
                    nomor = n
                    break
            potongan.append((token, nomor))
    return potongan

print("WARNA_SENSE siap:", WARNA_SENSE)

In [ ]:
# tabel HTML: legenda + kalimat lengkap guru & siswa yang sudah diwarnai
def buat_html_warna(kata, kode_soal, nama_siswa):
    gloss, sumber = ambil_kbbi_online(kata)
    baris_soal = data[data["Kode_Soal"] == kode_soal]
    kunci = str(baris_soal.iloc[0]["Kunci Guru"])
    jawab = str(baris_soal[baris_soal["Nama"] == nama_siswa].iloc[0]["Jawaban Siswa"])

    def kasih_warna(teks):
        html = ""
        for token, warna in pecah_warna(teks, kata):
            if warna == "TARGET":
                html += (f'<span style="background:{WARNA_TARGET};color:#fff;'
                         f'font-weight:bold;border-radius:3px;padding:0 3px">{token}</span>')
            elif warna is not None:
                html += (f'<span style="background:{WARNA_SENSE[warna]};color:#fff;'
                         f'font-weight:bold;border-radius:3px;padding:0 3px">{token}</span>')
            else:
                html += token.replace("\n", "<br>")
        return html

    def hitung_skor(teks):
        konteks = list(dict.fromkeys([k for k in bersihkan(teks) if k not in STOP]))
        skor = [sum(1 for k in konteks if k != kata and k in g.split()) for g in gloss]
        return skor, skor.index(max(skor)) + 1

    legenda = " ".join(
        f'<span style="background:{WARNA_SENSE[n]};color:#fff;padding:2px 7px;'
        f'border-radius:3px;margin:2px">sense {n} = {g[:32]}...</span>'
        for n, g in enumerate(gloss, 1))
    skor_g, pilih_g = hitung_skor(kunci)
    skor_s, pilih_s = hitung_skor(jawab)
    status = "SESUAI" if pilih_g == pilih_s else "TIDAK SESUAI"
    return f"""
<div style="border:1px solid #888;padding:12px;font-family:Arial;font-size:13px;line-height:2.1">
<h4 style="margin:2px 0">PENANDAAN WARNA kata '{kata}' (soal {kode_soal}) - sense dari {sumber}</h4>
<p><b>arti ke-n KBBI online:</b> {legenda}</p>
<p><span style="background:{WARNA_TARGET};color:#fff;font-weight:bold;padding:1px 6px;border-radius:3px">kata target</span>
&nbsp;| tanpa warna = tidak masuk gloss mana pun (tidak memberi skor)</p>
<p><b>KUNCI JAWABAN GURU</b> - skor {skor_g} -&gt; sense {pilih_g}:</p>
<p style="border:1px dashed #aaa;padding:6px">{kasih_warna(kunci)}</p>
<p><b>JAWABAN SISWA</b> ({nama_siswa}) - skor {skor_s} -&gt; sense {pilih_s}:</p>
<p style="border:1px dashed #aaa;padding:6px">{kasih_warna(jawab)}</p>
<p><b>KESIMPULAN:</b> sense_guru = {pilih_g}, sense_siswa = {pilih_s} -&gt; <b>{status}</b>
(kunci penilaian = word sense guru)</p>
</div>"""

print("fungsi buat_html_warna siap")

In [ ]:
from IPython.display import HTML
HTML_TEGANG  = buat_html_warna("tegang", 17, "Afif Ardliansyah Nur Holis")
HTML_MEMBRAN = buat_html_warna("membran", 20, "Nabilah Aushaf Ramadhani")
print("tabel penandaan warna ditampilkan di bawah (tegang & membran)")
HTML(HTML_TEGANG + HTML_MEMBRAN)

In [ ]:
# versi file xlsx: kata kena gloss ditebalkan & diwarnai di dalam sel (rich text)
from openpyxl import load_workbook
from openpyxl.cell.rich_text import TextBlock, CellRichText
from openpyxl.cell.text import InlineFont
from openpyxl.styles import Alignment, Font, PatternFill

WARNA_FONT = {n: w.lstrip("#") for n, w in WARNA_SENSE.items()}
WARNA_FONT_TARGET = WARNA_TARGET.lstrip("#")

def sel_rich(teks, kata_target):
    """kalimat -> CellRichText: kata kena gloss sense n = warna n & tebal."""
    runs = []
    for token, warna in pecah_warna(teks, kata_target):
        if warna == "TARGET":
            runs.append(TextBlock(InlineFont(b=True, u="single",
                                             color="FF" + WARNA_FONT_TARGET), token))
        elif warna is not None:
            runs.append(TextBlock(InlineFont(b=True,
                                             color="FF" + WARNA_FONT[warna]), token))
        else:
            runs.append(token.replace("\n", " "))
    return CellRichText(*runs)

print("fungsi sel_rich siap (contoh):", str(sel_rich("sel tegang dan kaku", "tegang"))[:90])

In [ ]:
# penulisan tabel warna ke sheet: judul, header, kunci guru, semua siswa
def tulis_tabel_warna(ws, kata, kode_soal):
    gloss, sumber = ambil_kbbi_online(kata)
    baris_soal = data[data["Kode_Soal"] == kode_soal]
    kunci = str(baris_soal.iloc[0]["Kunci Guru"])
    konteks_g = list(dict.fromkeys([k for k in bersihkan(kunci) if k not in STOP]))
    skor_g = [sum(1 for k in konteks_g if k != kata and k in g.split()) for g in gloss]
    sense_g = skor_g.index(max(skor_g)) + 1

    ws["A1"] = (f"PENANDAAN WARNA '{kata}' (soal {kode_soal}) - "
                f"sense = arti ke-n KBBI online ({sumber})")
    ws["A1"].font = Font(bold=True, size=12)
    for kolom, judul in [("A", "siapa (sense berapa)"),
                         ("B", "jawaban LENGKAP (kata berwarna = masuk sense itu)"),
                         ("C", "status vs guru")]:
        ws[kolom + "2"] = judul
        ws[kolom + "2"].font = Font(bold=True)
        ws[kolom + "2"].fill = PatternFill("solid", start_color="FFE0E0E0")

    b = 3
    ws.cell(row=b, column=1, value=f"KUNCI GURU (sense {sense_g})")
    ws.cell(row=b, column=2, value=sel_rich(kunci, kata))
    ws.cell(row=b, column=3, value="acuan (kunci)")
    b += 1
    for _, r in baris_soal.iterrows():
        if kata not in bersihkan(r["Jawaban Siswa"]):
            continue
        konteks_s = list(dict.fromkeys(
            [k for k in bersihkan(r["Jawaban Siswa"]) if k not in STOP]))
        skor_s = [sum(1 for k in konteks_s if k != kata and k in g.split())
                  for g in gloss]
        sense_s = skor_s.index(max(skor_s)) + 1
        ws.cell(row=b, column=1, value=f"{r['Nama']} (sense {sense_s})")
        ws.cell(row=b, column=2, value=sel_rich(str(r["Jawaban Siswa"]), kata))
        ws.cell(row=b, column=3, value="SESUAI" if sense_s == sense_g else "TIDAK SESUAI")
        b += 1
    return b, sense_g, gloss

In [ ]:
# legenda warna + catatan penilaian di bawah tabel
def tulis_legenda_warna(ws, b, kata, sense_g, gloss):
    b += 1
    ws.cell(row=b, column=1, value="KETERANGAN WARNA").font = Font(bold=True)
    b += 1
    for n, g in enumerate(gloss, 1):
        c = ws.cell(row=b, column=1, value=f"sense {n}")
        c.fill = PatternFill("solid", start_color="FF" + WARNA_FONT[n])
        c.font = Font(bold=True, color="FFFFFFFF")
        cb = ws.cell(row=b, column=2, value=f"arti ke-{n} KBBI online: {g}")
        cb.font = Font(bold=True, color="FF" + WARNA_FONT[n])
        b += 1
    for t in [
        "kata PINK bergaris bawah = kata target yang dianalisis.",
        "kata tebal berwarna = ada di gloss sense itu -> memberi skor ke sense itu.",
        "kata tanpa warna = tidak masuk gloss mana pun; kalau semua -> default sense 1.",
        "kalau 1 kata ada di lebih dari 1 gloss, dipakai sense pertama yang cocok.",
        "penilaian: SESUAI kalau sense siswa = sense guru (kunci = word sense guru).",
        f"cek langsung KBBI online: https://kbbi.raf555.dev/api/v1/entry/{kata}",
        f"versi csv per kata: penandaan_warna_{kata}.csv (kolom masuk_sense & warna).",
    ]:
        ws.cell(row=b, column=1, value=t)
        b += 1
    return b

In [ ]:
# gabungkan: buat sheet, isi tabel + legenda, rapihkan, simpan
def simpan_jawaban_berwarna(nama_file, kata, kode_soal):
    wb = load_workbook(nama_file)
    if "Jawaban_Berwarna" in wb.sheetnames:
        del wb["Jawaban_Berwarna"]
    ws = wb.create_sheet("Jawaban_Berwarna")
    for kolom, lebar in [("A", 28), ("B", 100), ("C", 14)]:
        ws.column_dimensions[kolom].width = lebar
    b, sense_g, gloss = tulis_tabel_warna(ws, kata, kode_soal)
    tulis_legenda_warna(ws, b, kata, sense_g, gloss)
    for row in ws.iter_rows(min_row=3, min_col=2, max_col=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    for row in ws.iter_rows(min_row=3, min_col=1, max_col=1):
        for c in row:
            c.alignment = Alignment(vertical="top")
    wb.save(nama_file)

simpan_jawaban_berwarna("studi_kasus_tegang.xlsx", "tegang", 17)
simpan_jawaban_berwarna("studi_kasus_membran.xlsx", "membran", 20)
print("disimpan : sheet 'Jawaban_Berwarna' di studi_kasus_tegang.xlsx & studi_kasus_membran.xlsx")

In [ ]:
# versi csv: 1 baris = 1 kata; warna dicatat sebagai kolom
# (csv tidak bisa menyimpan warna, jadi 'masuk_sense' & 'warna' jadi data)
WARNA_NAMA = {1: "kuning tua", 2: "oranye", 3: "biru", 4: "hijau",
              5: "ungu", 6: "merah muda", 7: "merah bata"}

def baris_warna(kata, siapa, status, teks, gloss):
    """list baris csv untuk 1 orang: kata target + kata konteks unik."""
    rows, urut, sudah = [], 0, set()
    for token, warna in pecah_warna(teks, kata):
        if token == kata:
            rows.append({"kata_target": kata, "siapa": siapa,
                         "status_vs_guru": status, "urutan": "-",
                         "kata": token, "masuk_sense": "TARGET",
                         "warna": "pink (kata target)",
                         "arti_sense_dari_KBBI_online": "-"})
        elif token.isalpha() and token not in STOP and token not in sudah:
            sudah.add(token)
            urut += 1
            rows.append({"kata_target": kata, "siapa": siapa,
                         "status_vs_guru": status, "urutan": urut,
                         "kata": token,
                         "masuk_sense": warna if warna is not None else "-",
                         "warna": WARNA_NAMA[warna] if warna is not None
                                  else "-",
                         "arti_sense_dari_KBBI_online":
                             gloss[warna - 1] if warna is not None else "-"})
    return rows

In [ ]:
# gabungkan kunci guru + semua siswa jadi satu tabel csv
def tabel_warna_csv(kata, kode_soal):
    """versi csv dari pewarnaan: 1 baris = 1 kata (target + kata konteks)."""
    gloss, _ = ambil_kbbi_online(kata)
    baris_soal = data[data["Kode_Soal"] == kode_soal]
    kunci = str(baris_soal.iloc[0]["Kunci Guru"])
    konteks_g = list(dict.fromkeys([k for k in bersihkan(kunci) if k not in STOP]))
    skor_g = [sum(1 for k in konteks_g if k != kata and k in g.split())
              for g in gloss]
    sense_g = skor_g.index(max(skor_g)) + 1
    rows = baris_warna(kata, f"KUNCI GURU (sense {sense_g})", "acuan (kunci)",
                       kunci, gloss)
    for _, r in baris_soal.iterrows():
        if kata not in bersihkan(r["Jawaban Siswa"]):
            continue
        konteks_s = list(dict.fromkeys(
            [k for k in bersihkan(r["Jawaban Siswa"]) if k not in STOP]))
        skor_s = [sum(1 for k in konteks_s if k != kata and k in g.split())
                  for g in gloss]
        sense_s = skor_s.index(max(skor_s)) + 1
        rows += baris_warna(kata, f"{r['Nama']} (sense {sense_s})",
                            "SESUAI" if sense_s == sense_g else "TIDAK SESUAI",
                            str(r["Jawaban Siswa"]), gloss)
    return pd.DataFrame(rows)

for kata_csv, kode_csv in [("tegang", 17), ("membran", 20)]:
    tabel = tabel_warna_csv(kata_csv, kode_csv)
    tabel.to_csv(f"penandaan_warna_{kata_csv}.csv", index=False, sep=";",
                 encoding="utf-8-sig")
    print(f"disimpan : penandaan_warna_{kata_csv}.csv "
          f"({len(tabel)} baris, pemisah ';')")

In [ ]:
# ringkasan hasil
t1 = (tabel_tegang["status"] == "SESUAI").sum()
t2 = (tabel_membran["status"] == "SESUAI").sum()
n_ok = (perbandingan["status"] == "SESUAI").sum()
print("=== RINGKASAN ===")
print(f"1. Kata ambigu (sense > 1)  : {len(kata_ambigu)} kata")
print(f"2. Akurasi simplified lesk  : {benar}/{len(df)} = {benar/len(df)*100:.1f}%")
print(f"3. Guru vs siswa            : {n} kata, {n_ok/n*100:.1f}% SESUAI (kunci = sense guru)")
print("4. Sumber sense             : KBBI online (kbbi.raf555.dev)")
print(f"5. Studi kasus              : tegang {len(tabel_tegang)} siswa ({t1} SESUAI), "
      f"membran {len(tabel_membran)} siswa ({t2} SESUAI)")
print(f"6. Gloss biologi vs umum    : {b_bio/t_bio*100:.1f}% vs {b_umum/t_umum*100:.1f}%")
print("7. File keluaran            : hasil_wsd_guru_vs_siswa.*, studi_kasus_tegang/membran.*, "
      "penandaan_warna_*.*")